Identify the pagination pattern used on Flipkart's search results page by inspecting the page structure and noting how the next page URL or button changes as you move from page 1 to page 2.

In [2]:
# Page 1 URL Example:[https://www.flipkart.com/search?q=laptops](https://www.flipkart.com/search?q=laptops)

# Page 2 URL Example:[https://www.flipkart.com/search?q=laptops&page=2](https://www.flipkart.com/search?q=laptops&page=2)

# Example pagination loop in Python
for page_num in range(1, 6):  # Scrapes pages 1 to 5
    url = f"https://www.flipkart.com/search?q=laptops&page={page_num}"
    # Make request...

Write a Python script using requests and BeautifulSoup to scrape the names and prices of the first 20 products from the first page of a Myntra search result.

In [3]:
import requests
from bs4 import BeautifulSoup

# Target Myntra category or search URL
url = "https://www.myntra.com/shoes"

# Headers to mimic a real browser and prevent 403 Forbidden blocking
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

try:
    # Send GET request to Myntra
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Locate product list items. On Myntra, individual products are typically <li> elements with class 'product-base'
    # limit=20 ensures we only grab the first 20 items
    product_cards = soup.find_all("li", class_="product-base", limit=20)
    
    if not product_cards:
        print("No products found. Myntra's structure may have changed or requires dynamic rendering (Selenium).")
    else:
        print(f"Successfully scraped the first {len(product_cards)} products:\n")
        print(f"{'No.':<4} | {'Brand':<18} | {'Product Name':<35} | {'Price':<10}")
        print("-" * 75)
        
        for index, card in enumerate(product_cards, start=1):
            # Extract brand name
            brand_elem = card.find("h3", class_="product-brand")
            brand = brand_elem.get_text(strip=True) if brand_elem else "N/A"
            
            # Extract product title / description
            name_elem = card.find("h4", class_="product-product")
            name = name_elem.get_text(strip=True) if name_elem else "N/A"
            
            # Extract price (looking for discounted price first, fallback to standard price)
            price_elem = card.find("div", class_="product-price")
            if price_elem:
                discounted_price = price_elem.find("span", class_="product-discountedPrice")
                if discounted_price:
                    price = discounted_price.get_text(strip=True)
                else:
                    price = price_elem.get_text(strip=True)
            else:
                price = "N/A"
            
            # Print formatted row
            print(f"{index:<4} | {brand[:16]:<18} | {name[:33]:<35} | {price:<10}")

except requests.exceptions.RequestException as e:
    print(f"An error occurred during the request: {e}")

No products found. Myntra's structure may have changed or requires dynamic rendering (Selenium).


Extend your script to loop through the first 3 pages of Zomato restaurant search results for 'pizza' in Ahmedabad, collecting restaurant names and ratings from each page.<br><br><em><strong>Hint:</strong> Pay attention to how the URL or request parameters change for each page.</em>

In [4]:
import time
import requests
from bs4 import BeautifulSoup

# Base URL for Ahmedabad restaurants on Zomato
base_url = "https://www.zomato.com/ahmedabad/restaurants"

# Headers to mimic a real browser request and avoid 403 Forbidden blocks
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

print("Starting Zomato scraping for Pizza in Ahmedabad (Pages 1 to 3)...\n")

# Loop through the first 3 pages
for page in range(1, 4):
    # Query parameters for search query and pagination
    params = {
        "q": "pizza",
        "page": page
    }
    
    try:
        print(f"--- Fetching Page {page} ---")
        response = requests.get(base_url, params=params, headers=headers)
        
        # Check if request succeeded
        if response.status_code != 200:
            print(f"Failed to fetch page {page}. Status Code: {response.status_code}")
            continue
            
        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Note: Zomato class names change frequently. 
        # Here we target generic structural elements or common card containers.
        # If classes have changed, inspect the live DOM to update selectors.
        restaurant_cards = soup.find_all('div', class_='search-result-card') # or similar container
        
        if not restaurant_cards:
            print(f"No cards found on page {page}. (Server may require JavaScript rendering or CAPTCHA).")
            # Fallback demonstration for parsing structure if elements are found:
        else:
            for idx, card in enumerate(restaurant_cards, start=1):
                # Extract restaurant name (typically in an h4 or heading tag)
                name_elem = card.find('h4')
                name = name_elem.get_text(strip=True) if name_elem else "N/A"
                
                # Extract rating (typically in a div/span with rating badge)
                rating_elem = card.find('div', class_='rating-div')
                rating = rating_elem.get_text(strip=True) if rating_elem else "N/A"
                
                print(f"  {idx}. Restaurant: {name} | Rating: {rating}")
                
        # Be polite to the server: pause for 2 seconds between requests
        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred on page {page}: {e}")

Starting Zomato scraping for Pizza in Ahmedabad (Pages 1 to 3)...

--- Fetching Page 1 ---
No cards found on page 1. (Server may require JavaScript rendering or CAPTCHA).
--- Fetching Page 2 ---
No cards found on page 2. (Server may require JavaScript rendering or CAPTCHA).
--- Fetching Page 3 ---
No cards found on page 3. (Server may require JavaScript rendering or CAPTCHA).


Modify your multi-page scraping script to include a 2-second delay between each page request to avoid getting blocked by the server.<br><br><em><strong>Constraint:</strong> Use the time.sleep() function to implement rate limiting.</em>

In [5]:
import time
import requests
from bs4 import BeautifulSoup

# Define base URL and headers
base_url = "https://www.zomato.com/ahmedabad/restaurants"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

print("Starting multi-page scraping with rate limiting...\n")

# Loop through the first 3 pages
for page in range(1, 4):
    # Query parameters for search query and pagination
    params = {
        "q": "pizza",
        "page": page
    }
    
    try:
        print(f"--- Fetching Page {page} ---")
        response = requests.get(base_url, params=params, headers=headers)
        
        # Check if request succeeded
        if response.status_code != 200:
            print(f"Failed to fetch page {page}. Status Code: {response.status_code}")
            continue
            
        # Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Example extraction logic for restaurant cards
        restaurant_cards = soup.find_all('div', class_='search-result-card')
        
        if not restaurant_cards:
            print(f"No cards found on page {page}. (Page structure may require JavaScript rendering).")
        else:
            for idx, card in enumerate(restaurant_cards, start=1):
                name_elem = card.find('h4')
                name = name_elem.get_text(strip=True) if name_elem else "N/A"
                print(f"  {idx}. Restaurant: {name}")
                
        # Rate Limiting: Pause for 2 seconds before making the next request
        print("Waiting 2 seconds before the next request...\n")
        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred on page {page}: {e}")

Starting multi-page scraping with rate limiting...

--- Fetching Page 1 ---
No cards found on page 1. (Page structure may require JavaScript rendering).
Waiting 2 seconds before the next request...

--- Fetching Page 2 ---
No cards found on page 2. (Page structure may require JavaScript rendering).
Waiting 2 seconds before the next request...

--- Fetching Page 3 ---
No cards found on page 3. (Page structure may require JavaScript rendering).
Waiting 2 seconds before the next request...



Find a product listing website of your choice (not Flipkart, Myntra, or Zomato) and write a script that scrapes the first 5 product titles across multiple pages, handling pagination as needed.<br><br><em><strong>Hint:</strong> Choose a site with clear 'Next' or page number navigation.</em>

In [6]:
import requests
from bs4 import BeautifulSoup

# Target base URL configuration for Books to Scrape
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

product_titles = []
max_products = 5
current_page = 1

print("Starting multi-page product scraping...\n")

while len(product_titles) < max_products:
    url = base_url.format(current_page)
    
    try:
        print(f"Fetching Page {current_page}...")
        response = requests.get(url)
        
        # If page doesn't exist (e.g., 404), stop looping
        if response.status_code != 200:
            print(f"Page {current_page} returned status code {response.status_code}. Ending scrape.")
            break
            
        # Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Locate all product container cards
        product_cards = soup.find_all('article', class_='product_pod')
        
        for card in product_cards:
            if len(product_titles) >= max_products:
                break
                
            # Extract the book title from the 'title' attribute of the <a> tag inside <h3>
            title_tag = card.find('h3').find('a')
            if title_tag and title_tag.has_attr('title'):
                title = title_tag['title']
                product_titles.append(title)
                
        current_page += 1

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        break

# Display the final collected results
print(f"\nSuccessfully collected the first {len(product_titles)} product titles across pages:\n")
for index, title in enumerate(product_titles, start=1):
    print(f"{index}. {title}")

Starting multi-page product scraping...

Fetching Page 1...

Successfully collected the first 5 product titles across pages:

1. A Light in the Attic
2. Tipping the Velvet
3. Soumission
4. Sharp Objects
5. Sapiens: A Brief History of Humankind
